In [1]:
import numpy as np 
import pandas as pd 
import math 

In [6]:
class GaussianNaiveBayes: 

    def __init__(self): 
        # Nothing to do here!
        pass 

    def fit(self , X , y): 
        # convert X and y into numpy array 
        X = np.asarray(X)
        y = np.asarray(y)

        # find all the unique classes
        self.classes_ = np.unique(y)
        # find the number of classes and features
        n_classes , n_features = len(self.classes_) , X.shape[1]

        # initialize two array to store the mean and variance of each classes of every features. 
        self.means_ = np.zeros(shape = (n_classes , n_features))
        self.variances_ = np.zeros(shape = (n_classes , n_features)) 
        # initialize another array to store priors(probability) of each class 
        self.priors_ = np.zeros(shape = n_classes)

        # now calculate prior , mean and variance for every class 
        for index , class_k in enumerate(self.classes_): 
            # find the rows where class(y) is class_k 
            Xk = X[y == class_k]

            # now find the mean , variance and prior for class class_k
            self.means_[index] = Xk.mean(axis = 0) # columnwise mean
            self.variances_[index] = Xk.var(axis = 0)
            self.priors_[index] = Xk.shape[0] / X.shape[0]
        return self
    
    def predict(self , X):
        X = np.asarray(X)
        predictions = []

        # do the prediction for each row 
        for row in X:  
            pred , _ = self._compute_log_terms(row)
            predictions.append(pred)
        return np.array(predictions)

    def _compute_log_terms(self , row): 
        """ 
        Compute log prior , normalization term and exponent terms for a single row.
        """
        x_row = np.asarray(row)
        log_posteriors = [] # for every class there will be a posteriors
        term_details = []

        eps = 1e-9
        # now calculate the posterior each class 
        for index , class_k in enumerate(self.classes_): 
            # term-01: log prior 
            term1_log_prior = np.log(self.priors_[index])

            # term-02: normalization 
            term2_norm = -0.5 * np.sum(np.log(2 * math.pi * self.variances_[index]) + eps) # adding epsilon to avoid zero division

            # term-03: exponent
            term3_exp = - np.sum((x_row - self.means_[index]) ** 2 / (2 * self.variances_[index] + eps))

            # total log-posterior(log-score) 
            log_posterior = term1_log_prior + term2_norm + term3_exp
            log_posteriors.append(log_posterior)

            term_details.append({
                "class": class_k, 
                "log_prior": term1_log_prior, 
                "normalization": term2_norm, 
                "exponent": term3_exp, 
                "log_posterior": log_posterior 
            })
        # now find the argmax y where log_posterior is max 
        predicted_class = self.classes_[np.argmax(log_posteriors)]
        return predicted_class , term_details

In [3]:
# sample dataset
X = np.array([[6.0, 3.0, 5.0],
              [5.0, 3.5, 1.4],
              [6.5, 3.2, 5.3],
              [5.2, 3.4, 1.5]])
y = np.array(['No', 'Yes', 'No', 'Yes']) 

In [7]:
gnb = GaussianNaiveBayes()
gnb.fit(X , y)

In [8]:
x_test = np.array([[6.0, 3.0, 5.0]])
prediction = gnb.predict(x_test)

In [9]:
prediction

array(['No'], dtype='<U2')

In [11]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [12]:
X , y = load_breast_cancer(return_X_y = True)
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [13]:
clf = GaussianNaiveBayes()
clf.fit(X_train , y_train)

In [14]:
y_pred = clf.predict(X_test)

In [15]:
accuracy_score(y_test , y_pred)

0.9649122807017544